# PyTorX - ResNet-18 on CIFAR-100

This notebook trains a ResNet-18 on CIFAR-100 using ReRAM crossbar-aware layers.

Architecture: Conv(3→64, 3×3) → [BasicBlock×2]×4 stages (64→128→256→512) → AvgPool → FC(512→100)

In [ ]:
import os
import shutil
import warnings

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torchvision import datasets, transforms

from python.torx.module.layer import crxb_Conv2d, crxb_Linear

## Configuration

In [ ]:
# Training hyperparameters
BATCH_SIZE = 128
TEST_BATCH_SIZE = 100
EPOCHS = 200
LR = 0.1
MOMENTUM = 0.9
WEIGHT_DECAY = 5e-4
SEED = 1
LOG_INTERVAL = 10
TEST_ONLY = False

# ReRAM crossbar parameters
CRXB_SIZE = 64
VDD = 3.3
GWIRE = 0.0357
GLOAD = 0.25
GMAX = 0.000333
GMIN = 0.000000333
IR_DROP = False
SCALER_DW = 1
ENABLE_NOISE = False
ENABLE_SAF = False
ENABLE_EC_SAF = False
FREQ = 10e6
TEMP = 300
QUANTIZE = 8
WEIGHT_PRECISION = None
NUM_CLASSES = 100

## Utilities

In [ ]:
def save_checkpoint(state, is_best, filename):
    torch.save(state, filename)
    if is_best:
        shutil.copyfile(filename, 'model_best.pth.tar')


class AverageMeter(object):
    """Computes and stores the average and current value"""

    def __init__(self):
        self.reset()

    def reset(self):
        self.val = 0
        self.avg = 0
        self.sum = 0
        self.count = 0

    def update(self, val, n=1):
        self.val = val
        self.sum += val * n
        self.count += n
        self.avg = self.sum / self.count

## Model Definition

In [ ]:
class BasicBlock(nn.Module):
    expansion = 1

    def __init__(self, in_planes, planes, crxb_cfg, stride=1):
        super(BasicBlock, self).__init__()
        self.conv1 = crxb_Conv2d(in_planes, planes, kernel_size=3, stride=stride,
                                 padding=1, bias=False, **crxb_cfg)
        self.bn1 = nn.BatchNorm2d(planes)
        self.conv2 = crxb_Conv2d(planes, planes, kernel_size=3, stride=1,
                                 padding=1, bias=False, **crxb_cfg)
        self.bn2 = nn.BatchNorm2d(planes)

        self.shortcut = nn.Sequential()
        if stride != 1 or in_planes != planes:
            self.shortcut = nn.Sequential(
                crxb_Conv2d(in_planes, planes, kernel_size=1, stride=stride,
                            bias=False, **crxb_cfg),
                nn.BatchNorm2d(planes)
            )

    def forward(self, x):
        out = F.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        out += self.shortcut(x)
        out = F.relu(out)
        return out


class ResNet18(nn.Module):
    """ResNet-18 for CIFAR-100 mapped onto ReRAM crossbar arrays."""

    def __init__(self, crxb_size, gmin, gmax, gwire, gload, vdd, ir_drop,
                 freq, temp, device, scaler_dw, enable_noise,
                 enable_SAF, enable_ec_SAF, num_classes=100,
                 quantize=8, weight_precision=None):
        super(ResNet18, self).__init__()

        crxb_cfg = dict(crxb_size=crxb_size, scaler_dw=scaler_dw,
                        gwire=gwire, gload=gload, gmax=gmax, gmin=gmin,
                        vdd=vdd, freq=freq, temp=temp,
                        enable_SAF=enable_SAF, enable_ec_SAF=enable_ec_SAF,
                        enable_noise=enable_noise, ir_drop=ir_drop,
                        device=device, quantize=quantize,
                        weight_precision=weight_precision)

        self.in_planes = 64

        self.conv1 = crxb_Conv2d(3, 64, kernel_size=3, stride=1, padding=1,
                                 bias=False, **crxb_cfg)
        self.bn1 = nn.BatchNorm2d(64)
        self.layer1 = self._make_layer(64, 2, stride=1, crxb_cfg=crxb_cfg)
        self.layer2 = self._make_layer(128, 2, stride=2, crxb_cfg=crxb_cfg)
        self.layer3 = self._make_layer(256, 2, stride=2, crxb_cfg=crxb_cfg)
        self.layer4 = self._make_layer(512, 2, stride=2, crxb_cfg=crxb_cfg)
        self.fc = crxb_Linear(512, num_classes, **crxb_cfg)

    def _make_layer(self, planes, num_blocks, stride, crxb_cfg):
        strides = [stride] + [1] * (num_blocks - 1)
        layers = []
        for s in strides:
            layers.append(BasicBlock(self.in_planes, planes, crxb_cfg, stride=s))
            self.in_planes = planes
        return nn.Sequential(*layers)

    def forward(self, x):
        out = F.relu(self.bn1(self.conv1(x)))
        out = self.layer1(out)
        out = self.layer2(out)
        out = self.layer3(out)
        out = self.layer4(out)
        out = F.avg_pool2d(out, 4)
        out = out.view(out.size(0), -1)
        out = self.fc(out)
        return F.log_softmax(out, dim=1)

## Training and Validation Functions

In [ ]:
def train(model, device, criterion, optimizer, train_loader, epoch):
    losses = AverageMeter()
    model.train()
    correct = 0

    for batch_idx, (data, target) in enumerate(train_loader):
        for name, module in model.named_modules():
            if isinstance(module, crxb_Conv2d) or isinstance(module, crxb_Linear):
                module._reset_delta()

        data, target = data.to(device), target.to(device)
        optimizer.zero_grad()
        output = model(data)
        loss = criterion(output, target)
        losses.update(loss.item(), data.size(0))
        loss.backward()
        optimizer.step()
        pred = output.max(1, keepdim=True)[1]
        correct += pred.eq(target.view_as(pred)).sum().item()

        if batch_idx % LOG_INTERVAL == 0:
            print('Train Epoch: {} [{}/{} ({:.0f}%)]\tLoss: {:.6f}'.format(
                epoch, batch_idx * len(data), train_loader.sampler.__len__(),
                       100. * batch_idx / len(train_loader), loss.item()))

    print('\nTrain set: Accuracy: {}/{} ({:.4f}%)\n'.format(
        correct, train_loader.sampler.__len__(),
        100. * correct / train_loader.sampler.__len__()))
    return losses.avg


def validate(model, device, criterion, val_loader):
    model.eval()
    test_loss = 0
    correct = 0
    with torch.no_grad():
        for batch_idx, (data, target) in enumerate(val_loader):
            data, target = data.to(device), target.to(device)
            output = model(data)
            test_loss += criterion(output, target).item()
            pred = output.max(1, keepdim=True)[1]
            correct += pred.eq(target.view_as(pred)).sum().item()

            if IR_DROP:
                print('\nTest set: Accuracy: {}/{} ({:.4f}%)\n'.format(
                    correct, val_loader.batch_sampler.__dict__['batch_size'] * (batch_idx + 1),
                             100. * correct / (val_loader.batch_sampler.__dict__['batch_size'] * (batch_idx + 1))))

        test_loss /= len(val_loader.dataset)
        print('\nTest set: Average loss: {:.4f}, Accuracy: {}/{} ({:.4f}%)\n'.format(
            test_loss, correct, val_loader.sampler.__len__(),
            100. * correct / val_loader.sampler.__len__()))
        return test_loss

## Setup

In [ ]:
if IR_DROP and (not TEST_ONLY):
    warnings.warn("We don't recommend training with IR drop, too slow!")

if IR_DROP and TEST_BATCH_SIZE > 150:
    warnings.warn("Reduce the batch size, IR drop is memory hungry!")

if not TEST_ONLY and ENABLE_NOISE:
    raise KeyError("Noise can cause unsuccessful training!")

use_cuda = torch.cuda.is_available()
torch.manual_seed(SEED)
device = torch.device("cuda" if use_cuda else "cpu")
print(f"Using device: {device}")

kwargs = {'num_workers': 2, 'pin_memory': True} if use_cuda else {}

transform_train = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize((0.5071, 0.4867, 0.4408),
                         (0.2675, 0.2565, 0.2761)),
])

transform_test = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5071, 0.4867, 0.4408),
                         (0.2675, 0.2565, 0.2761)),
])

train_loader = torch.utils.data.DataLoader(
    datasets.CIFAR100('../data', train=True, download=True,
                      transform=transform_train),
    batch_size=BATCH_SIZE, shuffle=True, **kwargs)

test_loader = torch.utils.data.DataLoader(
    datasets.CIFAR100('../data', train=False,
                      transform=transform_test),
    batch_size=TEST_BATCH_SIZE, shuffle=False, **kwargs)

model = ResNet18(crxb_size=CRXB_SIZE, gmax=GMAX, gmin=GMIN, gwire=GWIRE, gload=GLOAD,
                 vdd=VDD, ir_drop=IR_DROP, device=device, scaler_dw=SCALER_DW,
                 freq=FREQ, temp=TEMP, enable_SAF=ENABLE_SAF,
                 enable_noise=ENABLE_NOISE, enable_ec_SAF=ENABLE_EC_SAF,
                 num_classes=NUM_CLASSES, quantize=QUANTIZE,
                 weight_precision=WEIGHT_PRECISION).to(device)

optimizer = optim.SGD(model.parameters(), lr=LR, momentum=MOMENTUM,
                      weight_decay=WEIGHT_DECAY)
criterion = torch.nn.NLLLoss().to(device)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")

## Training Loop

In [ ]:
best_error = float('inf')
loss_log = []

if not TEST_ONLY:
    for epoch in range(EPOCHS):
        print("epoch {0}, and now lr = {1:.6f}\n".format(
            epoch, optimizer.param_groups[0]['lr']))
        train_loss = train(model=model, device=device, criterion=criterion,
                           optimizer=optimizer, train_loader=train_loader,
                           epoch=epoch)
        val_loss = validate(model=model, device=device,
                            criterion=criterion, val_loader=test_loader)

        scheduler.step()

        loss_log += [(epoch, train_loss, val_loss)]
        is_best = val_loss < best_error
        best_error = min(val_loss, best_error)

        filename = 'checkpoint_resnet18_cifar100_' + str(CRXB_SIZE) + '.pth.tar'
        save_checkpoint(state={
            'epoch': epoch + 1,
            'state_dict': model.state_dict(),
            'best_acc1': best_error,
            'optimizer': optimizer.state_dict(),
        }, is_best=is_best, filename=filename)
else:
    modelfile = 'checkpoint_resnet18_cifar100_' + str(CRXB_SIZE) + '.pth.tar'
    if os.path.isfile(modelfile):
        print("=> loading checkpoint '{}'".format(modelfile))
        checkpoint = torch.load(modelfile)
        model.load_state_dict(checkpoint['state_dict'])
        optimizer.load_state_dict(checkpoint['optimizer'])
        print("=> loaded checkpoint '{}'".format(modelfile))
        validate(model=model, device=device, criterion=criterion,
                 val_loader=test_loader)
    else:
        print(f"No checkpoint found at {modelfile}")

## Results Summary

In [ ]:
if loss_log:
    print(f"{'Epoch':>6} {'Train Loss':>12} {'Val Loss':>12}")
    print("-" * 32)
    for epoch, tl, vl in loss_log:
        print(f"{epoch:>6d} {tl:>12.6f} {vl:>12.6f}")
    print(f"\nBest validation loss: {best_error:.6f}")